In [4]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score


In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("SBERT loaded")


SBERT loaded


In [9]:
#load the data
def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():  # skip empty lines
                data.append(json.loads(line))
    return data

classification_data=load_jsonl("/content/synthetic_data_for_classification (3).jsonl")


In [13]:
train_anchor=[d["anchor_text"] for d in classification_data]
train_a = [d["text_a"] for d in classification_data]
train_b = [d["text_b"] for d in classification_data]

#binary labels: 1 closer to A,)=closer to B
y_train=[1 if d["text_a_is_closer"]==1 else 0 for d in classification_data]
print(len(train_anchor),len(y_train))

1900 1900


In [14]:
anchor_emb_train = model.encode(train_anchor, batch_size=32, show_progress_bar=True)
a_emb_train = model.encode(train_a, batch_size=32, show_progress_bar=True)
b_emb_train = model.encode(train_b, batch_size=32, show_progress_bar=True)


Batches:   0%|          | 0/60 [00:00<?, ?it/s]

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

In [17]:
#
X_train=np.hstack([
    np.abs(anchor_emb_train-a_emb_train)
,
    np.abs(anchor_emb_train-b_emb_train)
])
print("train feature shape:",X_train.shape)

train feature shape: (1900, 768)


In [18]:
#training the classifier
clf=LogisticRegression(max_iter=1000)
clf.fit(X_train,y_train)
print("classifier trained")

classifier trained


In [23]:
#testing on dev data

#load dev data
import json

dev_data = []

with open("/content/dev_track_a.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():              # skip empty lines
            dev_data.append(json.loads(line))


#extract dev texts and labels
dev_anchor = [d["anchor_text"] for d in dev_data]
dev_a = [d["text_a"] for d in dev_data]
dev_b = [d["text_b"] for d in dev_data]

y_dev = [1 if d["text_a_is_closer"] == 1 else 0 for d in dev_data]

#encode dev texts
anchor_emb_dev = model.encode(dev_anchor, batch_size=32, show_progress_bar=True)
a_emb_dev = model.encode(dev_a, batch_size=32, show_progress_bar=True)
b_emb_dev = model.encode(dev_b, batch_size=32, show_progress_bar=True)

#building dev feature
X_dev = np.hstack([
    np.abs(anchor_emb_dev - a_emb_dev),
    np.abs(anchor_emb_dev - b_emb_dev)
])

print("Dev feature shape:", X_dev.shape)




Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Dev feature shape: (200, 768)


In [24]:
y_pred = clf.predict(X_dev)

print("Accuracy:", accuracy_score(y_dev, y_pred))
print(classification_report(y_dev, y_pred))


Accuracy: 0.56
              precision    recall  f1-score   support

           0       0.55      0.58      0.56        99
           1       0.57      0.54      0.56       101

    accuracy                           0.56       200
   macro avg       0.56      0.56      0.56       200
weighted avg       0.56      0.56      0.56       200

